In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("alx-spark-02").getOrCreate()

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

weather_schema = StructType([
    StructField('county', StringType()),   StructField('station', StringType()),
    StructField('latitude', DoubleType()), StructField('longitude', DoubleType()),
    StructField('date', StringType()),     StructField('rain', DoubleType()),
    StructField('temp', DoubleType()),     StructField('wetb', DoubleType()),
    StructField('dewpt', DoubleType()),    StructField('vappr', DoubleType()),
    StructField('rhum', IntegerType()),    StructField('msl', DoubleType()),
    StructField('wdsp', IntegerType()),    StructField('wddir', IntegerType()),
    StructField('sun', DoubleType()),      StructField('vis', IntegerType()),
    StructField('clht', IntegerType()),    StructField('clamt', IntegerType())
])

raw = spark.read.csv("data/hrly_Irish_weather.csv", header=True, schema=weather_schema)
weather = raw.withColumn("date", F.to_timestamp("date", "dd-MMM-yyyy HH:mm"))
weather.write.mode("overwrite").parquet("data/weather_parquet")

In [5]:
working_df = spark.read.parquet("data/weather_parquet")
working_df.printSchema()
working_df.where(F.col("date").isNull()).count()   # must be 0

root
 |-- county: string (nullable = true)
 |-- station: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- rain: double (nullable = true)
 |-- temp: double (nullable = true)
 |-- wetb: double (nullable = true)
 |-- dewpt: double (nullable = true)
 |-- vappr: double (nullable = true)
 |-- rhum: integer (nullable = true)
 |-- msl: double (nullable = true)
 |-- wdsp: integer (nullable = true)
 |-- wddir: integer (nullable = true)
 |-- sun: double (nullable = true)
 |-- vis: integer (nullable = true)
 |-- clht: integer (nullable = true)
 |-- clamt: integer (nullable = true)



0

In [6]:
column_map = {"wdsp": "wind_speed_kt", "rhum": "humidity_pct"}   # old → new
mapped = working_df
for old, new in column_map.items():
    mapped = mapped.withColumnRenamed(old, new)

In [7]:
working_df.groupBy(working_df.columns).count().where("count > 1").show()   # identical rows
working_df.groupBy("date", "station").count().where("count > 1").show()    # same key twice

+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+-----+
|county|station|latitude|longitude|date|rain|temp|wetb|dewpt|vappr|rhum|msl|wdsp|wddir|sun|vis|clht|clamt|count|
+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+-----+
+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+-----+

+----+-------+-----+
|date|station|count|
+----+-------+-----+
+----+-------+-----+



In [8]:
working_df.where(F.col('rain') < 0).show() #Rain should be positive. Let's see if we have any negative records

+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+
|county|station|latitude|longitude|date|rain|temp|wetb|dewpt|vappr|rhum|msl|wdsp|wddir|sun|vis|clht|clamt|
+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+
+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+



In [9]:
working_df.where(~F.col('temp').between(-20, 40)).show() # ~ operator means 'everything except'.

+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+
|county|station|latitude|longitude|date|rain|temp|wetb|dewpt|vappr|rhum|msl|wdsp|wddir|sun|vis|clht|clamt|
+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+
+------+-------+--------+---------+----+----+----+----+-----+-----+----+---+----+-----+---+---+----+-----+



In [10]:
working_df.where(F.col('dewpt') > F.col('temp')).groupBy('county').count().show()

+---------+-----+
|   county|count|
+---------+-----+
|Tipperary|   10|
|     Mayo|   54|
|   Galway|    5|
|    Cavan|    4|
|    Clare|    1|
|   Dublin|  103|
|     Cork|  106|
|   Carlow|   24|
|Roscommon|   31|
|Westmeath|   20|
|    Sligo|    2|
|    Meath|    7|
|    Kerry|    2|
+---------+-----+



In [11]:
rules = {
    "vappr": F.col("vappr") >= 0,
    "rhum":  F.col("rhum").between(0, 100),
    "msl":   F.col("msl").between(900, 1100),   # "close to 1013.25": real-world range
    "wdsp":  F.col("wdsp").between(0, 220),
    "wddir": F.col("wddir").between(0, 360),
    "sun":   F.col("sun").between(0, 24),
    "vis":   F.col("vis") >= 0,
    "clht":  F.col("clht").between(0, 999),
    "clamt": F.col("clamt").between(0, 9),
}

for col_name, valid in rules.items():
    invalid = working_df.where(~valid).count()
    nulls = working_df.where(F.col(col_name).isNull()).count()
    print(col_name, "| invalid:", invalid, "| nulls:", nulls)

vappr | invalid: 0 | nulls: 187484
rhum | invalid: 15 | nulls: 163615
msl | invalid: 0 | nulls: 73283
wdsp | invalid: 0 | nulls: 311009
wddir | invalid: 0 | nulls: 326368
sun | invalid: 0 | nulls: 2816527
vis | invalid: 0 | nulls: 2855416
clht | invalid: 0 | nulls: 2816545
clamt | invalid: 0 | nulls: 2816545


In [12]:
working_df.where(~F.col("rhum").between(0, 100)).groupBy("rhum").count().show()

+----+-----+
|rhum|count|
+----+-----+
| -14|   15|
+----+-----+



In [13]:
column_map = {
    "rain": "rain_mm",          "temp": "temp_c",
    "wetb": "wet_bulb_temp_c",  "dewpt": "dew_point_c",
    "vappr": "vapour_pressure_hpa", "rhum": "humidity_pct",
    "msl": "sea_level_pressure_hpa", "wdsp": "wind_speed_kt",
    "wddir": "wind_direction_deg",   "sun": "sun_hours",
    "vis": "visibility_m",      "clht": "cloud_height",
    "clamt": "cloud_amount_oktas",
}

mapped = working_df
for old, new in column_map.items():
    mapped = mapped.withColumnRenamed(old, new)
mapped.printSchema()

root
 |-- county: string (nullable = true)
 |-- station: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- rain_mm: double (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- wet_bulb_temp_c: double (nullable = true)
 |-- dew_point_c: double (nullable = true)
 |-- vapour_pressure_hpa: double (nullable = true)
 |-- humidity_pct: integer (nullable = true)
 |-- sea_level_pressure_hpa: double (nullable = true)
 |-- wind_speed_kt: integer (nullable = true)
 |-- wind_direction_deg: integer (nullable = true)
 |-- sun_hours: double (nullable = true)
 |-- visibility_m: integer (nullable = true)
 |-- cloud_height: integer (nullable = true)
 |-- cloud_amount_oktas: integer (nullable = true)



In [14]:
F.when(F.col("wetb").isin(49.0, 49.9, 99.0), None).otherwise(F.col("wetb"))

Column<'CASE WHEN in(wetb, 49.0, 49.9, 99.0) THEN NULL ELSE wetb END'>

In [15]:
sentinels = {"wetb": [49.0, 49.9, 99.0], "dewpt": [92.4], "rhum": [-14]}

clean_df = working_df
for col_name, bad_values in sentinels.items():
    clean_df = clean_df.withColumn(
        col_name,
        F.when(F.col(col_name).isin(bad_values), None).otherwise(F.col(col_name))
    )

print(working_df.count(), clean_df.count())       # must be equal: no rows lost
clean_df.where(F.col("rhum") == -14).count()      # must be 0

4660423 4660423


0

In [16]:
clean_df = (clean_df
    .withColumn("is_humid", F.col("rhum") > 80)
    .withColumn("clamt_missing", F.col("clamt").isNull())   # flag FIRST
    .fillna({"clamt": 0})                                   # then fill
)

total = clean_df.count()
complete = clean_df.dropna().count()
print(total, complete, total - complete, "rows would be lost with dropna()")

4660423 1804797 2855626 rows would be lost with dropna()


In [17]:
clean_df = clean_df.orderBy("date", "station") \
                   .withColumn("idx", F.monotonically_increasing_id())

clean_df.select(F.max("idx"), F.count("*")).show()

+-----------+--------+
|   max(idx)|count(1)|
+-----------+--------+
|60130196175| 4660423|
+-----------+--------+



In [46]:
customers = spark.createDataFrame([
    ("Amina Benali", "amina@mail.com", "0612345689", "Khouribga"),
    ("Youssef Idrissi", "youssef@mail.com", "0698765412", "Casablanca"),
], ["name", "email", "phone", "city"])
safe_customers = (customers
    .drop("name")
    .withColumn("email_masked", F.concat(F.substring("email", 1, 1),              
             F.lit("****@"),                           
             F.substring_index("email", "@", -1)))
    .withColumn("phone", F.concat(F.lit("********"), F.substring("phone", -2, 2)))
    .drop("email")
)
safe_customers.show(truncate=False)

+----------+----------+--------------+
|phone     |city      |email_masked  |
+----------+----------+--------------+
|********89|Khouribga |a****@mail.com|
|********12|Casablanca|y****@mail.com|
+----------+----------+--------------+



In [48]:
safe_customers = safe_customers.withColumn(
    "email_hashed",
    F.sha2(F.col("email_masked"), 256))

safe_customers.show(truncate=False)

+----------+----------+--------------+----------------------------------------------------------------+
|phone     |city      |email_masked  |email_hashed                                                    |
+----------+----------+--------------+----------------------------------------------------------------+
|********89|Khouribga |a****@mail.com|9621636900c2cff594ad01edfe2e2e79da9f991d952b259b6228e9b83fd896e4|
|********12|Casablanca|y****@mail.com|6cd13a310dd05286c554cdde7c91c7f1bb31ec37413fc073cff61553d9c7b019|
+----------+----------+--------------+----------------------------------------------------------------+



In [49]:
convert_dict = {
    'county': 'County', 'station': 'Weather Station',
    'latitude': 'Latitude', 'longitude': 'Longitude', 'date': 'Date',
    'rain': 'Rain (mm)', 'temp': 'Temperature (°C)',
    'wetb': 'Wet Bulb Air Temperature (°C)', 'dewpt': 'Dew Point',
    'vappr': 'Vapour Pressure (hPa)', 'rhum': 'Relative Humidity (%)',
    'msl': 'Mean Sea Level Pressure (hPa)', 'wdsp': 'Mean Hourly Wind Speed (kt)',
    'wddir': 'Predominant Hourly Wind Direction (degrees)', 'sun': 'Sun (hours)',
    'vis': 'Visibility (m)', 'clht': 'Cloud Ceiling Height',
    'clamt': 'Cloud Amount (Oktas)', 'idx': 'Index',
    'is_humid': 'Is Humid (>80%)', 'clamt_missing': 'Cloud Amount Missing',
}
convert_dict["is_humid"] = "Is Humid (>80%)"
convert_dict["clamt_missing"] = "Cloud Amount Missing"

output_df = clean_df.withColumn("date", F.col("date").cast("string"))
for c in output_df.columns:
    output_df = output_df.withColumnRenamed(c, convert_dict.get(c, c))
output_df.printSchema()

root
 |-- County: string (nullable = true)
 |-- Weather Station: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Date: string (nullable = true)
 |-- Rain (mm): double (nullable = true)
 |-- Temperature (°C): double (nullable = true)
 |-- Wet Bulb Air Temperature (°C): double (nullable = true)
 |-- Dew Point: double (nullable = true)
 |-- Vapour Pressure (hPa): double (nullable = true)
 |-- Relative Humidity (%): integer (nullable = true)
 |-- Mean Sea Level Pressure (hPa): double (nullable = true)
 |-- Mean Hourly Wind Speed (kt): integer (nullable = true)
 |-- Predominant Hourly Wind Direction (degrees): integer (nullable = true)
 |-- Sun (hours): double (nullable = true)
 |-- Visibility (m): integer (nullable = true)
 |-- Cloud Ceiling Height: integer (nullable = true)
 |-- Cloud Amount (Oktas): integer (nullable = false)
 |-- Is Humid (>80%): boolean (nullable = true)
 |-- Cloud Amount Missing: boolean (nullable = false)

In [27]:
yearly_temp = (clean_df
    .withColumn("year", F.year("date"))
    .groupBy("year")
    .pivot("county")
    .agg(F.round(F.avg("temp"), 1))
    .orderBy("year")
)
yearly_temp.show(truncate=False)

+----+------+-----+-----+----+-------+------+------+-----+----+-----+---------+-----+---------+---------+-------+
|year|Carlow|Cavan|Clare|Cork|Donegal|Dublin|Galway|Kerry|Mayo|Meath|Roscommon|Sligo|Tipperary|Westmeath|Wexford|
+----+------+-----+-----+----+-------+------+------+-----+----+-----+---------+-----+---------+---------+-------+
|1990|NULL  |NULL |11.0 |10.6|10.1   |10.2  |NULL  |11.3 |10.1|NULL |NULL     |NULL |NULL     |9.5      |NULL   |
|1991|NULL  |NULL |10.6 |9.7 |9.6    |9.6   |NULL  |10.9 |9.8 |NULL |NULL     |NULL |NULL     |9.1      |NULL   |
|1992|NULL  |NULL |10.5 |9.6 |9.6    |9.7   |NULL  |10.8 |9.6 |NULL |NULL     |NULL |NULL     |9.0      |NULL   |
|1993|NULL  |NULL |10.2 |6.3 |9.4    |9.5   |NULL  |10.8 |9.5 |NULL |NULL     |NULL |NULL     |8.7      |NULL   |
|1994|NULL  |NULL |10.6 |9.7 |9.6    |9.5   |NULL  |11.0 |9.7 |NULL |NULL     |NULL |NULL     |9.1      |NULL   |
|1995|NULL  |NULL |11.3 |9.0 |10.2   |10.1  |NULL  |11.5 |10.2|NULL |NULL     |NULL |NUL

### Structured streaming

In [50]:
stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

counts = (stream
    .withColumn("is_even", F.col("value") % 2 == 0)
    .groupBy("is_even").count())

q = (counts.writeStream
    .outputMode("complete")
    .format("memory")            # keeps results in a table you can query
    .queryName("even_counts")
    .start())

In [51]:
spark.sql("SELECT * FROM even_counts").show()

+-------+-----+
|is_even|count|
+-------+-----+
|   true|   20|
|  false|   20|
+-------+-----+



In [52]:
spark.sql("SELECT * FROM even_counts").show()

+-------+-----+
|is_even|count|
+-------+-----+
|   true|  148|
|  false|  147|
+-------+-----+



In [57]:
spark.sql("SELECT * FROM even_counts").show()
# The numbers grow: that’s the unbounded table.
q.stop() #to finish it

+-------+-----+
|is_even|count|
+-------+-----+
|   true|  620|
|  false|  620|
+-------+-----+



In [58]:
stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()

counts = (stream
    .withColumn("is_even", F.col("value") % 2 == 0)
    .groupBy("is_even").count())

q = (counts.writeStream
    .outputMode("append") #changed complete to append => error
    .format("memory")
    .queryName("even_counts")
    .start())

AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_OPERATION] Invalid streaming output mode: append. This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets. SQLSTATE: 42KDE

In [59]:
spark.version

'4.2.0'

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 36944)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/socketserver.py", line 766, in __init__
    self.handle()
    ~~~~~~~~~~~^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 329, in handle
    poll(accum_updates)
    ~~~~^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/pyspark/accumu